# Customer Churn Prediction & Retention Analytics
End-to-end EDA, classification, risk scoring and retention segmentation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
print(df.shape)
display(df.head())


## Data Cleaning

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df.isna().sum())
df = df.dropna(subset=["TotalCharges"]).copy()
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)
print(df.shape)


## Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Churn", ax=axes[0])
contract = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
contract.plot(kind="bar", ax=axes[1])
axes[0].set_title("Churn Distribution")
axes[1].set_title("Churn by Contract")
plt.tight_layout()


In [ ]:
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Churn")
plt.show()
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, kde=True, element="step")
plt.title("Tenure Distribution by Churn")
plt.show()


## Modeling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X = df.drop(columns=["Churn", "ChurnFlag"])
y = df["ChurnFlag"]
cat = X.select_dtypes(include="object").columns.tolist()
num = X.select_dtypes(exclude="object").columns.tolist()
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=400, random_state=42, class_weight="balanced", min_samples_leaf=2, n_jobs=-1)
}
for name, model in models.items():
    pipe = Pipeline([("prep", prep), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1]
    print(name, {
        "accuracy": round(accuracy_score(y_test,pred),4),
        "precision": round(precision_score(y_test,pred),4),
        "recall": round(recall_score(y_test,pred),4),
        "f1": round(f1_score(y_test,pred),4),
        "roc_auc": round(roc_auc_score(y_test,prob),4)
    })


## Business Risk Segmentation

In [ ]:
print("Run src/train_model.py to generate the scored customer file and Power BI dataset.")
print("Risk thresholds used in the project: High >= 0.70, Medium 0.40-0.69, Low < 0.40.")
